In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import xarray as xr
import seaborn as sns
sns.set_theme("notebook",style="dark")

In [ ]:
import os
from pathlib import Path

from qgsw.logging.core import getLogger, setup_root_logger
from qgsw.specs import defaults


torch.backends.cudnn.deterministic = True

ROOT_PATH = Path(os.getcwd()).parent


specs=defaults.get()

logger=getLogger(__name__)
setup_root_logger(1)

In [ ]:
dt = 7200
n_steps_per_cycle = 250
separation = 250
comparison_interval = 1
n_cycles = 12

In [ ]:
from qgsw.configs.core import Configuration
from qgsw.forcing.wind import WindForcing


config = Configuration.from_toml("../output/g5k/param_optim/_config.toml")

H = config.model.h
g_prime = config.model.g_prime
H1, H2 = H[0], H[1]
g1, g2 = g_prime[0], g_prime[1]

H1_, H2_ = H1,H2
g1_,g2_ = g1,g2

beta_plane = config.physics.beta_plane
bottom_drag_coef = config.physics.bottom_drag_coefficient
slip_coef = config.physics.slip_coef

wind = WindForcing.from_config(
    config.windstress,
    config.space,
    config.physics,
)
tx, ty = wind.compute()
b = 4

In [ ]:
from qgsw.spatial.core.discretization import SpaceDiscretization3D


space = SpaceDiscretization3D.from_config(
    config.space,
    config.model,
)
dx, dy = space.dx, space.dy

## Initial condition

In [ ]:
from qgsw.fields.variables.tuples import UVH
from qgsw.masks import Masks
from qgsw.models.qg.stretching_matrix import compute_A
from qgsw.models.qg.uvh.projectors.core import QGProjector
from qgsw.utils import covphys


P = QGProjector(
    A=compute_A(H=H, g_prime=g_prime),
    H=H.unsqueeze(-1).unsqueeze(-1),
    space=space,
    f0=beta_plane.f0,
    masks=Masks.empty(
        nx=config.space.nx,
        ny=config.space.ny,
    ),
)
uvh0 = UVH.from_file("../output/g5k/param_optim/_data_startup.pt")
psi_start = P.compute_p(covphys.to_cov(uvh0, dx, dy))[0] / beta_plane.f0

### Full-domain model

In [ ]:
from qgsw.models.qg.psiq.core import QGPSIQ


model_3l = QGPSIQ(
    space_2d=space.remove_h(),
    H=H,
    beta_plane=config.physics.beta_plane,
    g_prime=g_prime,
)
model_3l.set_wind_forcing(tx, ty)
model_3l.masks = Masks.empty_tensor(
    model_3l.space.nx,
    model_3l.space.ny,
    device=specs["device"],
)
model_3l.bottom_drag_coef = bottom_drag_coef
model_3l.slip_coef = slip_coef
model_3l.dt = dt
y0 = model_3l.y0


In [ ]:
from collections.abc import Callable
from pathlib import Path
from typing import Any, Generic, TypeVar

from matplotlib import pyplot as plt
import numpy as np
from qgsw.analysis.qg_model import ModelWrapper, ModelsManager
from qgsw.decomposition.coefficients import DecompositionCoefs
from qgsw.decomposition.core import build_basis_from_params_dict
from qgsw.eNATL60.wind import compute_windstress
from qgsw.models.qg.psiq.core import QGPSIQCore
from qgsw.models.qg.psiq.modified.forced import QGPSIQForced, QGPSIQForcedSurfML, QGPSIQRGPsi2TransportDR
from qgsw.scripts.boundaries import extract_psi_bc, extract_wide_psi_bc
from qgsw.scripts.eNATL60 import filter_streamfunction
from qgsw.scripts.output import OutputFolder, build_output_folder
from qgsw.scripts.pv import build_pv_funcs
from qgsw.scripts.setups.model import setup_model
from qgsw.spatial.core.discretization import SpaceDiscretization2D
from qgsw.utils.interpolation import QuadraticInterpolation
from qgsw.utils.reshaping import crop

T = TypeVar("T", bound=QGPSIQCore)

class ModelWrapperOBC(ModelWrapper[T],Generic[T]):
    model:T
    indices:tuple[int,int,int,int]
    no_wind:bool

    def __init__(self, output_folder: Path | str | None, data_dt: float=dt) -> None:
        super().__init__(output_folder, data_dt)

    @property
    def indices(self) -> tuple[int,int,int,int]:
        return self._indices

    @indices.setter
    def indices(self, indices:tuple[int,int,int,int]) -> None:
        self._indices = indices

    def parse_params(self, output_folder:OutputFolder) -> dict[str, Any]:
        params=super().parse_params(output_folder)
        self.indices = params["indices"]
        self.no_wind = params["no_wind"]
        return params

    def init_model(self,space:SpaceDiscretization2D) -> T:

        imin,imax,jmin,jmax = self.indices
        
        space_slice = space.slice(imin, imax + 1, jmin, jmax + 1)

        space_slice_wide = space.slice(
            imin - b, imax + b + 1, jmin - b, jmax + b + 1
        )

        self.model =  setup_model(
            self.model_type,
            space_slice,
            torch.tensor([H1_, H2_], **specs),
            torch.tensor([g1_, g2_], **specs),
            beta_plane,
            self.dt,
            config.physics.slip_coef,
        )
        self.model.y0 = y0
        if not self.no_wind:
            self.model.set_wind_forcing(
                tx[imin:imax, jmin : jmax + 1], ty[imin : imax + 1, jmin:jmax]
            )
        y_wide = space_slice_wide.q.xy.y[0, :].unsqueeze(0)
        beta_effect_wide = beta_plane.beta * (y_wide - y0)
        self.build_pv = build_pv_funcs(
            dx, dy, beta_plane.f0, beta_effect_wide, b, space_slice_wide.nx
        )

    def check_indices(self, indices:tuple[int,int,int,int]) -> None:
        if self._output is None:
            return
        data_indices = self.indices
        assert indices == data_indices, "Mismatching indices."
    
    def setup(self,sfs:torch.Tensor|xr.DataArray, times:list[torch.Tensor]) -> None:
        try:
            super().setup(sfs,times)
        except KeyError:
            return

        if self.model.A.shape[1] == 1:
            A11 = self.model.A[:1, :1]
            A12 = torch.zeros_like(A11)
        else:
            A11 = self.model.A[:1, :1]
            A12 = self.model.A[:1, 1:2]

        compute_q, compute_q_bcs = self.build_pv(A11, A12)   


        psi_bcs = [extract_psi_bc(psi,b) for psi in sfs]
        psi_bcs_w = [extract_wide_psi_bc(psi,b, clone=True) for psi in sfs]

        psi0 = sfs[0]
        q_bcs = [compute_q_bcs(p1) for p1 in psi_bcs_w]
        self.model.set_psiq(crop(psi0,b), crop(compute_q(psi0),b-1))
        self.model.set_boundary_maps(QuadraticInterpolation(times, psi_bcs), QuadraticInterpolation(times, q_bcs))
        if self.remember_psis:
            self.psis.append(self.model.psi.clone())

class ModelsManagerOBC(ModelsManager[ModelWrapperOBC[QGPSIQCore]]):

    def check_indices(self, indices:tuple[int,int,int,int]) -> None:
        self.loop_over_models(lambda mw:mw.check_indices(indices))

In [ ]:
from qgsw.solver.finite_diff import grad_perp


def perform_cycle(indices:tuple[int,int,int,int],models:ModelsManagerOBC,comparison_interval:int=1) -> torch.Tensor:

    imin,imax,jmin,jmax = indices
    psi_slices_w = [
            slice(imin - b, imax + b + 1),
            slice(jmin - b, jmax + b + 1),
        ]
    extract_psi_w = lambda psi: psi[..., psi_slices_w[0], psi_slices_w[1]]

    psis = [extract_psi_w(model_3l.psi[:,:2])]
    times = [model_3l.time.item()]
    
    for _ in range(1,n_steps_per_cycle):
        model_3l.step()
        psis.append(extract_psi_w(model_3l.psi[:,:2]))
        times.append(model_3l.time.item())
        
    psis_ref = torch.stack(psis,dim=0)
    
    models.new_cycle()

    models.setup(psis_ref[:,:,:1,...], times)

    models.reset_time()

    u_refs,v_refs = grad_perp(psis_ref)
    u_refs/=dy
    v_refs/=dx

    models.compute_loss(crop(psis_ref[0],b))
    models.compute_uv_losses(crop(u_refs[0],b),crop(v_refs[0],b))
    for n in range(1,n_steps_per_cycle):
        models.step()
        if n % comparison_interval == 0:
            models.compute_loss(crop(psis_ref[int(n//comparison_interval)],b))
            models.compute_uv_losses(crop(u_refs[int(n//comparison_interval)],b),crop(v_refs[int(n//comparison_interval)],b))
    for _ in range(separation):
        model_3l.step()

    return psis_ref


In [ ]:
import gc

In [ ]:
from qgsw.logging.utils import box, step
from qgsw.models.qg.psiq.core import QGPSIQ

folder = Path("../output/g5k/param_optim")

indices = [32, 96, 256, 384 ]
imin,imax,jmin,jmax = indices
rg = ModelWrapperOBC[QGPSIQ](None)
rg.label = "RG-0"
rg.color="black"
rg.indices=indices
rg.no_wind=False
rg.dt=7200

rgf = ModelWrapperOBC[QGPSIQForced](folder /"QG3L_RGF-20261022_173236-122798f4")
rgf.dt=7200
rgf.color ='brown'
rgf.linestyle = "solid"
rgf.label = "RG-F"

rgsi = ModelWrapperOBC[QGPSIQRGPsi2TransportDR](folder /"QG3L_RGSI-20261022_075508-7757181e")
rgsi.dt=7200
rgsi.color = "navy"
rgsi.linestyle = "solid"
rgsi.label = "RG-SI"

rgfsi = ModelWrapperOBC[QGPSIQForcedSurfML](folder /"QG3L_RGFSI-20261021_181955-8e02ac21")
rgfsi.dt=7200
rgfsi.color = "orange"
rgfsi.linestyle = "solid"
rgfsi.label = "RG-FSI"

models = ModelsManagerOBC(
    rg,
    rgf,
    rgsi,
    rgfsi,
)
space_slice = space.remove_h().slice(imin, imax + 1, jmin, jmax + 1)
models.add_gradient_RMSE(space_slice)
models.add_MSC(space_slice)
models.add_PSDE(space_slice)
models.check_indices(indices)
models.init_models(space.remove_h())
models.set_wind_forcing(tx[imin:imax, jmin : jmax + 1], ty[imin : imax + 1, jmin:jmax])
models.save_params = True

models.remember_psis(mode=False)

c_start = 0
c_end = 12

model_3l.set_psi(psi_start)

for _ in range(0, c_start):
    for _ in range(1,n_steps_per_cycle):
        model_3l.step()
    models.new_cycle()
    for _ in range(separation):
        model_3l.step()

gc.collect()
for c in range(c_start, c_end):
    torch.cuda.reset_peak_memory_stats()

    times = np.arange(n_steps_per_cycle)*dt+c*n_steps_per_cycle*dt

    psis_ref = perform_cycle(indices,models)

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="="))


In [ ]:
from copy import deepcopy


losses_z1 = {
    mw.label:deepcopy(mw.losses) for mw in models.model_wrappers
}

In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show
# forced_g10000.linestyle = "solid"

# Plots

# rg.show = True
show_grad = True and ("uv_rmse" in models.uv_losses)

cycle = None

fig,axs = plots.subplots(1+show_grad,1,figsize=(22,(5)*(1+show_grad)))
fig.suptitle(indices)
plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0],cycle=cycle)
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[1,0],cycle=cycle)
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

ref = crop(psis_ref[-1][0,0],b)

vmax = ref.abs().max()

plots.imshow(ref,ax=axs[0,0],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)==0)

for i,mw in enumerate(models.model_wrappers):
    plots.imshow(mw.model.psi[0,0],ax=axs[0,i+1],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i)
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots
from qgsw.solver.finite_diff import laplacian

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

vorticities = laplacian(psis_ref,dx,dy)

ref = crop(vorticities[-1][0,0],b+1)/beta_plane.f0

vmax = ref.abs().quantile(0.99)

plots.imshow(ref,robust=True,ax=axs[0,0],show_cbar=len(models.model_wrappers)==0,vmin=-vmax,vmax=vmax)

for i,mw in enumerate(models.model_wrappers):
    plots.imshow(crop(mw.model.vorticity[0,0]/beta_plane.f0,1),ax=axs[0,i+1],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i)
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,4)


for i,mw in enumerate(models.model_wrappers):
    axs[0,0].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0])[0].cpu(),**mw.plot_kwargs)
    axs[0,1].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0])[-1].cpu(),**mw.plot_kwargs)
    axs[0,2].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0]).mean(axis=0).cpu(),**mw.plot_kwargs)
    axs[0,3].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0])[120:].mean(axis=0).cpu(),**mw.plot_kwargs)
axs[0,-1].legend()
plots.clamp_ylims(0,1,axs[0,0])
plots.clamp_ylims(0,1,axs[0,1])
plots.clamp_ylims(0,1,axs[0,2])
plots.clamp_ylims(0,1,axs[0,3])
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,4)


for i,mw in enumerate(models.model_wrappers):
    axs[0,0].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0])[0].cpu(),**mw.plot_kwargs)
    axs[0,1].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0])[-1].cpu(),**mw.plot_kwargs)
    axs[0,2].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0]).mean(axis=0).cpu(),**mw.plot_kwargs)
    axs[0,3].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0])[200:].mean(axis=0).cpu(),**mw.plot_kwargs)
axs[0,-1].legend()
plots.clamp_ylims(0,1,axs)
plt.tight_layout()
plt.show()

## Z3: [112 176 64 192]

In [ ]:
from qgsw.logging.utils import box, step
from qgsw.models.qg.psiq.core import QGPSIQ

folder = Path("../output/g5k/param_optim")

indices = [112, 176, 64, 192 ]
imin,imax,jmin,jmax = indices
rg = ModelWrapperOBC[QGPSIQ](None)
rg.no_wind = False
rg.label = "RG-0"
rg.color="black"
rg.indices=indices
rg.dt=7200

rgf = ModelWrapperOBC[QGPSIQForced](folder /"QG3L_RGF-20261022_173312-01f4935c")
rgf.dt=7200
rgf.color ='brown'
rgf.linestyle = "solid"
rgf.label = "RG-F"

rgsi = ModelWrapperOBC[QGPSIQRGPsi2TransportDR](folder /"QG3L_RGSI-20261021_181342-8f38153b")
rgsi.dt=7200
rgsi.color = "navy"
rgsi.linestyle = "solid"
rgsi.label = "RG-SI"

rgfsi = ModelWrapperOBC[QGPSIQForcedSurfML](folder /"QG3L_RGFSI-20261022_075549-4f706f32")
rgfsi.dt=7200
rgfsi.color = "orange"
rgfsi.linestyle = "solid"
rgfsi.label = "RG-FSI"

models = ModelsManagerOBC(
    rg,
    rgf,
    rgsi,
    rgfsi,
)
space_slice = space.remove_h().slice(imin, imax + 1, jmin, jmax + 1)
models.add_gradient_RMSE(space_slice)
models.add_MSC(space_slice)
models.add_PSDE(space_slice)
models.check_indices(indices)
models.init_models(space.remove_h())
models.set_wind_forcing(tx[imin:imax, jmin : jmax + 1], ty[imin : imax + 1, jmin:jmax])
models.save_params = True

models.remember_psis(mode=False)

c_start = 0
c_end = 12

model_3l.reset_time()
model_3l.set_psi(psi_start)

for _ in range(0, c_start):
    for _ in range(1,n_steps_per_cycle):
        model_3l.step()
    models.new_cycle()
    for _ in range(separation):
        model_3l.step()

gc.collect()
for c in range(c_start, c_end):
    torch.cuda.reset_peak_memory_stats()

    psis_ref = perform_cycle(indices,models)

    torch.cuda.empty_cache()
    gc.collect()

    max_mem = torch.cuda.max_memory_allocated() / 1024 / 1024
    msg_mem = f"Cycle {step(c + 1, n_cycles)} | Max memory allocated: {max_mem:.1f} MB."
    logger.info(box(msg_mem, style="="))


In [ ]:
from copy import deepcopy


losses_z2 = {
    mw.label:deepcopy(mw.losses) for mw in models.model_wrappers
}

In [ ]:
from matplotlib import pyplot as plt

from qgsw import plots

# To Show
# forced_g10000.linestyle = "solid"
# Plots

# rg.show = True
show_grad = True
cycle = None

fig,axs = plots.subplots(1+show_grad,1,figsize=(22,(5)*(1+show_grad)))
fig.suptitle(indices)
plots.set_rowtitles(["RMSE"]+show_grad*[ "Gradient RMSE"] ,axs=axs)
models.plot_loss(loss_name="rmse",ax=axs[0,0],cycle=cycle)
plots.clamp_ylims(0,1,axs[0,0])
axs[0,0].legend(loc="upper left",prop={'size': 8})
if show_grad:
    models.plot_loss(loss_name="uv_rmse",ax=axs[1,0],cycle=cycle)
    plots.clamp_ylims(0,1,axs[1,0])
    axs[1,0].legend(loc="upper left",prop={'size': 8})

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

ref = crop(psis_ref[-2][0,0],b)

vmax = ref.abs().max()

plots.imshow(ref,ax=axs[0,0],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)==0)

for i,mw in enumerate(models.model_wrappers):
    plots.imshow(mw.model.psi[0,0],ax=axs[0,i+1],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i)
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots
from qgsw.solver.finite_diff import laplacian
from qgsw.spatial.core.grid_conversion import interpolate

fig,axs = plots.subplots(1,len(models.model_wrappers)+1)

coltitles = ["Reference"]

vorticities = laplacian(psis_ref, dx,dy)

ref = interpolate(vorticities[-1][0,0])

vmax = ref.abs().max()

plots.imshow(ref,robust=True,ax=axs[0,0],show_cbar=len(models.model_wrappers)==0,vmin=-vmax,vmax=vmax)

for i,mw in enumerate(models.model_wrappers):
    plots.imshow(crop(mw.model.vorticity[0,0],1),ax=axs[0,i+1],vmin=-vmax,vmax=vmax,show_cbar=len(models.model_wrappers)-1==i)
    coltitles.append(mw.label)
plots.set_coltitles(coltitles,axs=axs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,4)


for i,mw in enumerate(models.model_wrappers):
    axs[0,0].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0])[0].cpu(),**mw.plot_kwargs)
    axs[0,1].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0])[-1].cpu(),**mw.plot_kwargs)
    axs[0,2].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0]).mean(axis=0).cpu(),**mw.plot_kwargs)
    axs[0,3].plot(models.msc.kr.cpu(),torch.stack(mw.losses["msc"][0])[120:].mean(axis=0).cpu(),**mw.plot_kwargs)
plt.tight_layout()
plt.show()

In [ ]:
from qgsw import plots

fig,axs = plots.subplots(1,4)


for i,mw in enumerate(models.model_wrappers):
    axs[0,0].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0])[0].cpu(),**mw.plot_kwargs)
    axs[0,1].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0])[-1].cpu(),**mw.plot_kwargs)
    axs[0,2].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0]).mean(axis=0).cpu(),**mw.plot_kwargs)
    axs[0,3].plot(models.psd.kr.cpu(),torch.stack(mw.losses["psd_err"][0])[200:].mean(axis=0).cpu(),**mw.plot_kwargs)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import Size, Divider
import matplotlib.ticker as ticker
from math import sqrt

gratio = (1+sqrt(5))/2


fig = plt.figure(dpi=150)

# Define fixed sizes in inches
pad_left  = Size.Fixed(0.8)
ax_width  = Size.Fixed(5.0)
gap       = Size.Fixed(0.1)
pad_right = Size.Fixed(0.3)

row_height = Size.Fixed(5 / gratio)
row_gap    = Size.Fixed(0.4)   # vertical gap between the two rows

h = [pad_left, ax_width, gap, ax_width, pad_right]
v = [
    Size.Fixed(0.6),   # bottom margin
    row_height,        # bottom row (row 1)
    row_gap,           # gap between rows
    row_height,        # top row (row 2)
    Size.Fixed(0.4),   # top margin
]

divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)

# Bottom row  (ny=1)
ax1_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))
ax2_bot = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=1),
                       sharey=ax1_bot)

# Top row  (ny=3)
ax1_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=3),
                        sharex=ax1_bot)
ax2_top = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=3, ny=3),
                       sharey=ax1_top, sharex=ax2_bot)

times = np.arange(n_steps_per_cycle) * dt

# ── plotting helper ───────────────────────────────────────────────────────────
# def plot_on(ax_summer, ax_winter, losses_dict):

for mw in [rg, rgf, rgsi, rgfsi]:
    if not mw.show:
        continue
    t = times / 3600 / 24
    if mw.label in losses_z1:
        ls = torch.stack([torch.tensor(l, **specs) for l in losses_z1[mw.label]["rmse"]])
        not_nan = ~torch.isnan(ls).any(dim=1)
        ms   = torch.mean(ls[not_nan], dim=0)
        stds = torch.std(ls[not_nan],  dim=0)
        
        m, s = ms.cpu().numpy(), stds.cpu().numpy()
        ax1_top.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
        ax1_top.plot(t, m, **mw.plot_kwargs)
    if mw.label in losses_z2:
        ls = torch.stack([torch.tensor(l, **specs) for l in losses_z2[mw.label]["rmse"]])
        not_nan = ~torch.isnan(ls).any(dim=1)
        ms   = torch.mean(ls[not_nan], dim=0)
        stds = torch.std(ls[not_nan],  dim=0)
        m, s = ms.cpu().numpy(), stds.cpu().numpy()
        ax2_top.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
        ax2_top.plot(t, m, **mw.plot_kwargs)

for mw in [rg, rgf, rgsi,rgfsi]:
    if not mw.show:
        continue
    t = times / 3600 / 24
    if mw.label in losses_z1:
        ls = torch.stack([torch.tensor(l, **specs) for l in losses_z1[mw.label]["uv_rmse"]])
        not_nan = ~torch.isnan(ls).any(dim=1)
        ms   = torch.mean(ls[not_nan], dim=0)
        stds = torch.std(ls[not_nan],  dim=0)
        
        m, s = ms.cpu().numpy(), stds.cpu().numpy()
        ax1_bot.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
        ax1_bot.plot(t, m, **mw.plot_kwargs)
    if mw.label in losses_z2:
        ls = torch.stack([torch.tensor(l, **specs) for l in losses_z2[mw.label]["uv_rmse"]])
        not_nan = ~torch.isnan(ls).any(dim=1)
        ms   = torch.mean(ls[not_nan], dim=0)
        stds = torch.std(ls[not_nan],  dim=0)
        m, s = ms.cpu().numpy(), stds.cpu().numpy()
        ax2_bot.fill_between(t, m - s, m + s, alpha=0.2, color=mw.color)
        ax2_bot.plot(t, m, **mw.plot_kwargs)

# ── shared spine / tick style ─────────────────────────────────────────────────
def style_ax(ax, hide_yticks=False, hide_xticks=False):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["bottom"].set_bounds(0, 20)
    ax.spines["left"].set_bounds(0, 1)
    ax.spines["bottom"].set_position(("outward", 5))
    ax.spines["left"].set_position(("outward", 5))
    ax.tick_params(axis='both', labelsize="large")
    ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f'))
    if hide_yticks:
        ax.tick_params(labelleft=False)
    if hide_xticks:
        ax.tick_params(labelbottom=False)
        ax.spines["bottom"].set_visible(False)

style_ax(ax1_bot)
style_ax(ax2_bot, hide_yticks=True)
style_ax(ax1_top, hide_xticks=True)          # no x-axis on top row
style_ax(ax2_top, hide_yticks=True, hide_xticks=True)

# ── labels & titles ───────────────────────────────────────────────────────────
ax1_top.set_title("Z1", fontsize="x-large")
ax2_top.set_title("Z2",  fontsize="x-large")

ax1_bot.set_xlabel("Time [days]", fontsize="large")
ax2_bot.set_xlabel("Time [days]", fontsize="large")

ax1_top.set_ylabel("Stream function nRMSE", fontsize="large")
ax1_bot.set_ylabel("Velocity nRMSE", fontsize="large")

ax1_top.legend(fontsize="large")
plots.clamp_ylims(0, 1, ax1_top)
plots.clamp_ylims(0, 1, ax1_bot)
plots.set_ylims(0, None, ax1_bot)
plots.set_ylims(0, None, ax1_top)

# fig.savefig("../output/images/enatl60_all_qg3l_rmses.svg", bbox_inches="tight")